# Build 03-02 · Reweight mitigation — the corrector, per axis, on the exported training split

Runs in the **analysis `.venv`** (`python3` kernel), loads no model. Reads the
**treatment-joined training material** built by 03_01 (`corrector_targets` kind), runs the four
schemes, and writes one corrected target per axis into `src/data/real/mitigation/`. Retraining is
**03_03_retrain.ipynb**, on each version's own kernel.

```
inputs/features_<v>_<split>.parquet   +   mitigation/inputs/corrector_targets_<v>_<split>.parquet
   ──▶  §3 ReweightCorrector per axis (analysis env)
   ──▶  mitigation/<v>_corrected_<split>_<tag>.parquet  (+ _meta.json)
   ──▶  03_03_retrain.ipynb  (env-v2 / env-v3 kernels, separately)
```

The axes (full parameter definition in `src/mitigator/corrector/reweight.py`):

| scheme | garage rows | model-scrapped rows (U) |
|---|---|---|
| naive | (observed, 1) | (1, 1) — the contaminated baseline |
| rarity | (observed, m_c) | (1, m_c) — Axis A only, labels kept |
| transport | (observed, 1) | (1, g) + (0, 1−g) — Axis B soft split |
| pnu | (observed, m_c) | (1, g·m_c) + (0, (1−g)·m_c) — combined |

U is the **recorded `decision`**, never τ — naive/transport are identical under both τ modes and
run once; only rarity/pnu run twice (`regime` = per-row `config.threshold_on(DECIDER, date)`,
`fixed` = one scalar, read off this split's own boundary unless `TAU_FIXED` is set).

**Per-version feasibility** (thesis `tab:scheme-feasibility`): **v3** has score + decision (v2
serving log) → all four schemes. **v2** has decision only (vehicle-status file; the v1-era
deciding score is destroyed) → **naive/transport only** — §3 skips rarity/pnu automatically when
the input carries no `score`. v1 is out of scope (pre-model labels; no retraining).

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import threshold
from mitigator.corrector.reweight import ReweightCorrector

pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC: every knob of this run lives HERE. Paths resolve through config.
VERSION = "v3"        # "v2" runs naive/transport only (no deciding score — see header)
SPLIT = "train"       # one of config.SPLITS[VERSION] — the split 03_03 will retrain on
DECIDER = "v2"        # whose regime record resolves tau_i for the rarity band (v3's labels)
ID_COL = "claim_id"

FEATURES_PATH = config.split_path("processed_inputs", VERSION, SPLIT)
TARGETS_PATH = config.split_path("corrector_targets", VERSION, SPLIT)   # built by 03_01

# the axes: (scheme, tau_mode). naive/transport ignore tau -> run once.
RUNS = [("naive", None), ("rarity", "regime"), ("rarity", "fixed"),
        ("transport", None), ("pnu", "regime"), ("pnu", "fixed")]
BAND_H, CLIP_LO, CLIP_HI = 0.01, 0.25, 4.0
TAU_FIXED = None      # fixed mode: None -> read_off on this split (min scrap score, pooled eras)


def run_tag(scheme: str, mode: str | None) -> str:
    """Filename tag for one axis run, e.g. 'transport', 'rarity_regime'."""
    return scheme if mode is None else scheme + "_" + mode


def corrected_path(tag: str) -> Path:
    """config's corrected kind + split, with the axis tag before the extension."""
    base = config.split_path("corrected", VERSION, SPLIT)
    return base.with_name(base.stem + "_" + tag + base.suffix)


print("features :", FEATURES_PATH)
print("targets  :", TARGETS_PATH)
print("out      :", corrected_path("<tag>"))

insepct best BAND_H, CLIP_LO, CLIP_HI 

In [ ]:
# code implementaion 

## §2 — load & audit

`corrector_targets` arrives with the treatment already joined (03_01 dropped the rows its source
never covered — counts in that file's `_meta.json`). Here: canonical checks, P/N/U, and — when a
score column exists — the per-era boundary audit (`threshold.read_off` **per regime**, never
pooled) and the band-occupancy power gate for the rarity scheme's cell 1.

In [ ]:
# §2a — load + canonical checks
features = pd.read_parquet(FEATURES_PATH)
assert TARGETS_PATH.is_file(), f"{TARGETS_PATH} missing — run 03_01_corrector_inputs first"
corr_targets = pd.read_parquet(TARGETS_PATH)

need = [ID_COL, "date", "observed", "decision"]
miss = [c for c in need if c not in corr_targets.columns]
assert not miss, f"corrector_targets is missing {miss} — rebuild it with 03_01"
assert ID_COL in features.columns
HAS_SCORE = "score" in corr_targets.columns

corr_targets = corr_targets.copy()
corr_targets["date"] = pd.to_datetime(corr_targets["date"])
d = corr_targets["date"]
print(f"{VERSION} {SPLIT}: {len(corr_targets):,} rows, {d.min():%Y-%m-%d} -> {d.max():%Y-%m-%d}"
      f" | score column: {HAS_SCORE}")
if not HAS_SCORE:
    print("no deciding score -> rarity/pnu will be SKIPPED (v2 reality; thesis tab:scheme-feasibility)")

In [ ]:
# §2b — P/N/U, and (score only) per-era boundary audit + band occupancy
dec_col = corr_targets["decision"].astype(int)
obs_col = corr_targets["observed"].astype(int)
P = int(((dec_col == 0) & (obs_col == 1)).sum())
N = int(((dec_col == 0) & (obs_col == 0)).sum())
U = int((dec_col == 1).sum())
bad = int(((dec_col == 1) & (obs_col == 0)).sum())
warn = f"  !! scrapped-but-observed=0: {bad}" if bad else ""
print(f"P={P:,}  N={N:,}  U={U:,} ({U / len(corr_targets):.2%}){warn}")

if HAS_SCORE:
    days = d.dt.normalize()
    tau_lut = {day: config.threshold_on(DECIDER, str(day.date())) for day in days.unique()}
    tau_row = days.map(tau_lut).astype(float)
    brks = [pd.Timestamp(b["date"]) for b in config.breaks(DECIDER)]
    era = pd.cut(d, [pd.Timestamp.min] + brks + [pd.Timestamp.max], right=False)
    rows = []
    for e, g in corr_targets.groupby(era, observed=True):
        gd = g["decision"].astype(int)
        r = {"era": str(e), "tau_declared": float(tau_row[g.index].iloc[0]),
             "n": len(g), "n_scrapped": int(gd.sum())}
        if 0 < gd.sum() < len(g):
            ro = threshold.read_off(g)
            r |= {"applied_tau": ro["tau"], "deterministic": ro["deterministic"],
                  "overlap_rows": ro["overlap_rows"]}
        rows.append(r)
    display(pd.DataFrame(rows))
    for brk in config.spans_a_break(DECIDER, str(d.min().date()), str(d.max().date())):
        print("regime break inside this split:", brk)

    in_band = (dec_col == 0) & (corr_targets["score"] >= tau_row - BAND_H)
    print(f"garage rows in the band [tau-{BAND_H}, .): {int(in_band.sum()):,} "
          f"(repairable: {int((in_band & (obs_col == 0)).sum()):,})  <- rarity cell 1; gate on this n")
else:
    print("(no score column -> era audit and band gate not applicable)")

## §3 — run the corrector, one file per axis

`feature_cols` comes from `config.model_features(VERSION)` — never "every column except
claim_id" (the exported matrix carries the target, and v3's its own predictions). transport/pnu
outputs hold each U claim **twice** (the (1, g) / (0, 1−g) halves); `retrain.py`'s join expands
the feature row to match, so 03_03 needs no special handling. Runs whose requirements the input
cannot meet (rarity/pnu without a score) are skipped with a printed reason, not errored.

In [ ]:
# §3 — corrector per axis -> corrected parquet + meta sidecar
FEATURE_COLS = config.model_features(VERSION)

diag_rows = []
for scheme, mode in RUNS:
    tag = run_tag(scheme, mode)
    if scheme in ("rarity", "pnu") and not HAS_SCORE:
        print(f"{tag:16s} SKIPPED — needs the deciding score, absent for {VERSION}")
        continue
    corr = ReweightCorrector(scheme=scheme, tau_mode=mode or "regime", decider=DECIDER,
                             tau=TAU_FIXED, band_h=BAND_H, clip_lo=CLIP_LO, clip_hi=CLIP_HI,
                             id_col=ID_COL)
    out, diag = corr.correct(features, corr_targets, feature_cols=FEATURE_COLS)
    p = corrected_path(tag)
    p.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(p, index=False)
    p.with_name(p.stem + "_meta.json").write_text(
        json.dumps({"version": VERSION, "split": SPLIT,
                    "features": str(FEATURES_PATH), "targets": str(TARGETS_PATH), **diag},
                   indent=2),
        encoding="utf-8")
    diag_rows.append({"run": tag, **{k: v for k, v in diag.items() if not isinstance(v, dict)}})
    print(f"{tag:16s} -> {p.name}")
display(pd.DataFrame(diag_rows).set_index("run"))

## §4 — what got written, and what happens next

- `src/data/real/mitigation/<v>_corrected_<split>_<tag>.parquet` — `claim_id + label + weight`
  per axis, plus a `_meta.json` sidecar (scheme, τ mode, cell table, transport diagnostics).
- Next: **03_03_retrain.ipynb** on the version's own kernel (env-v2 / env-v3, separately) —
  it discovers these files by name and retrains one model per axis, then scores the splits.

Caveats to carry: 03_01 already dropped treatment-uncovered rows once, for every consumer alike;
the `fixed` τ arm reads one pooled boundary across eras **by design** (the
clean-single-threshold sensitivity), while `regime` is the faithful arm; if the band count in
§2b is a handful of rows, the rarity scheme's cell-1 up-weighting rests on those few rows —
gate on n.